# 第 13 章：二叉树模型

本章每一个数字背后的代码。**只用标准库**：整章都是格点上的算术，
不用 NumPy 写出来，可以让每一步都看得见，而不是被向量化掉。

| 节 | 复现什么 |
|---|---|
| 1 | 一步树、对冲与套利（例 13.1） |
| 2 | 真实世界的路线，以及它为什么是循环的（例 13.2） |
| 3 | 状态价格（例 13.3） |
| 4 | 三个状态，以及价格区间（例 13.4） |
| 5 | 两步，以及全程跟踪对冲（例 13.5） |
| 6 | 一份美式看跌期权与提前行权溢价（例 13.6） |
| 7 | 由波动率构建一棵树（例 13.7） |
| 8 | 美式期货看涨期权及其提前行权（例 13.8） |
| 9 | 收敛，以及布莱克–斯科尔斯–默顿极限（图 13.6） |
| 10 | 提前行权边界（图 13.5） |

每个单元都把算出的值列在本章所列的值旁边，
数字一旦漂移，就会显现出来。

表头保留英文：中文字符在等宽终端里是双宽的，而 Python 的 `%-10s` 按字符数计。

In [1]:
from math import exp, sqrt


def show(label, got, printed, tol=6e-4):
    """把算出来的值印在本章所印的值旁边。"""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-44s %12.6f   book %10.4f   %s" % (label, got, printed, flag))


S0, Su, Sd, K, r, dt = 20.0, 22.0, 18.0, 21.0, 0.04, 0.25
print("一步树: S0=%.0f -> {%.0f, %.0f}, K=%.0f, r=%.0f%%, dt=%.2f"
      % (S0, Su, Sd, K, 100 * r, dt))

一步树: S0=20 -> {22, 18}, K=21, r=4%, dt=0.25


## 1. 一步，以及为它定价的对冲：例 13.1

delta 的选法使得对冲后的组合在两个状态下支付相同。一旦如此，
该组合就是无风险的，必须赚取 $r$，而这一过程没有用到任何概率。
以 0.60 美元而不是按其价值卖出这份看涨期权，在*每一个*状态下都留下利润；
正因如此，它才是套利，而不只是一笔好赌注。

In [2]:
fu, fd = max(Su - K, 0.0), max(Sd - K, 0.0)
delta = (fu - fd) / (Su - Sd)
cash = (fd - delta * Sd) * exp(-r * dt)          # borrowing that completes the hedge
f = delta * S0 + cash

show("delta", delta, 0.25)
show("call value f", f, 0.545)

# the replicating portfolio pays what the call pays, in both states
up = delta * Su + cash * exp(r * dt)
dn = delta * Sd + cash * exp(r * dt)
show("replicating portfolio, up state", up, fu)
show("replicating portfolio, down state", dn, fd)

quote = 0.60
show("arbitrage profit at a %.2f quote" % quote, (quote - f) * exp(r * dt), 0.056)

delta                                            0.250000   book     0.2500   ok
call value f                                     0.544776   book     0.5450   ok
replicating portfolio, up state                  1.000000   book     1.0000   ok
replicating portfolio, down state                0.000000   book     0.0000   ok
arbitrage profit at a 0.60 quote                 0.055779   book     0.0560   ok


## 2. 真实世界的路线：例 13.2

认真做一遍，以说明它为什么被放弃。真实概率给出期望收益并不难；
它所需要的贴现率才是这个计算无法提供的，
因为要知道它，就要先知道所求的期权价格本身。

In [3]:
q_real = 0.7
ES = q_real * Su + (1 - q_real) * Sd
show("expected stock price", ES, 20.80)
show("expected return over the step, %", 100 * (ES / S0 - 1), 4.0)
show("expected call payoff", q_real * fu + (1 - q_real) * fd, 0.70)

# the required return the option would need, once the answer is already known.
# 本章是除以四舍五入后的 0.545，这里照做，并把两者都印出来。
EF = q_real * fu + (1 - q_real) * fd
show("implied required return, on f = 0.545, %", 100 * (EF / 0.545 - 1), 28.4, tol=3e-3)
print("%-44s %12.6f" % ("同一算式，用未四舍五入的 f", 100 * (EF / f - 1)))

expected stock price                            20.800000   book    20.8000   ok
expected return over the step, %                 4.000000   book     4.0000   ok
expected call payoff                             0.700000   book     0.7000   ok
implied required return, on f = 0.545, %        28.440367   book    28.4000   ok
同一算式，用未四舍五入的 f                                  28.493238


## 3. 状态价格：例 13.3

$\psi_u$ 与 $\psi_d$ 是今天为各状态下 1 美元所付的价格。
它们就是贴现后的风险中性概率；一旦它们存在，
这棵树上的每一份索取权都由一次点积定出价格，不需要再论证什么。

In [4]:
p = (exp(r * dt) - Sd / S0) / (Su / S0 - Sd / S0)
psi_u = exp(-r * dt) * p
psi_d = exp(-r * dt) * (1 - p)
show("risk-neutral probability p", p, 0.5503)
show("state price psi_u", psi_u, 0.5448)
show("state price psi_d", psi_d, 0.4453)

# 用复制上行状态的 1 美元来验证 psi_u
d_rep = (1.0 - 0.0) / (Su - Sd)
b_rep = (0.0 - d_rep * Sd) * exp(-r * dt)
show("B in that replication", b_rep, -4.4552)
show("cost today, which is psi_u", d_rep * S0 + b_rep, 0.5448)

# 现在不用再做任何论证，就给另一项资产定价
show("claim paying 3 up / 7 down", 3 * psi_u + 7 * psi_d, 4.7512)

risk-neutral probability p                       0.550251   book     0.5503   ok
state price psi_u                                0.544776   book     0.5448   ok
state price psi_d                                0.445274   book     0.4453   ok
B in that replication                           -4.455224   book    -4.4552   ok
cost today, which is psi_u                       0.544776   book     0.5448   ok
claim paying 3 up / 7 down                       4.751246   book     4.7512   ok


## 4. 三个状态：例 13.4

加进一个中间状态，复制就失败了：三个方程，两个未知数。
留下来的是一个与无套利相容的价格*区间*，
这就是第二基本定理的雏形，在此提前登场。

In [5]:
# psi must price the bond and the stock, and stay positive
#   psi1 + psi2 + psi3 = exp(-r dt)          (the bond)
#   22 psi1 + 20 psi2 + 18 psi3 = 20         (the stock)
# one free parameter remains; sweep it and read off the call's price range
lo, hi = None, None
steps = 200000
for i in range(1, steps):
    psi2 = exp(-r * dt) * i / steps
    psi1 = (20.0 - 18.0 * (exp(-r * dt) - psi2) - 20.0 * psi2) / (22.0 - 18.0)
    psi3 = exp(-r * dt) - psi1 - psi2
    if psi1 > 0 and psi2 > 0 and psi3 > 0:
        c = 1.0 * psi1                        # the call pays $1 only in the top state
        lo = c if lo is None else min(lo, c)
        hi = c if hi is None else max(hi, c)
print("该看涨期权的无套利价格区间: %.4f 到 %.4f" % (lo, hi))
print("复制方程组不相容，因此得不到唯一价格")

该看涨期权的无套利价格区间: 0.0995 到 0.5448
复制方程组不相容，因此得不到唯一价格


## 5. 两步，以及全程跟踪对冲：例 13.5

倒推法给出价格；这个例子要说明的是，对冲确实成立。
交易者卖出看涨期权，买入 $\Delta_A$ 股，其余借入，中途再平衡一次，
最后恰好拿到应付的收益。

In [6]:
u, d = 1.1, 0.9
p2 = (exp(r * dt) - d) / (u - d)
S = {"A": S0, "B": S0 * u, "C": S0 * d,
     "BB": S0 * u * u, "BC": S0 * u * d, "CC": S0 * d * d}
f_BB, f_BC, f_CC = (max(S[k] - K, 0.0) for k in ("BB", "BC", "CC"))
f_B = exp(-r * dt) * (p2 * f_BB + (1 - p2) * f_BC)
f_C = exp(-r * dt) * (p2 * f_BC + (1 - p2) * f_CC)
f_A = exp(-r * dt) * (p2 * f_B + (1 - p2) * f_C)

show("payoff at BB", f_BB, 3.20)
show("value at node B", f_B, 1.7433)
show("value at node A", f_A, 0.9497)

dA = (f_B - f_C) / (S["B"] - S["C"])
dB = (f_BB - f_BC) / (S["BB"] - S["BC"])
show("delta at A", dA, 0.4358)
show("delta at B", dB, 0.7273)

borrow = dA * S["A"] - f_A
show("borrowed at A", borrow, 7.7667)
show("debt one step later", borrow * exp(r * dt), 7.8448)
show("shares worth, at node B", dA * S["B"], 9.5881)

payoff at BB                                     3.200000   book     3.2000   ok
value at node B                                  1.743282   book     1.7433   ok
value at node A                                  0.949698   book     0.9497   ok
delta at A                                       0.435821   book     0.4358   ok
delta at B                                       0.727273   book     0.7273   ok
borrowed at A                                    7.766714   book     7.7667   ok
debt one step later                              7.844771   book     7.8448   ok
shares worth, at node B                          9.588053   book     9.5881   ok


## 6. 美式看跌期权：例 13.6

在每个节点上，价值取“继续持有”与“行权”二者中较大的一个。
行权只在节点 C 处最优，别处都不是；这既给出了期权价格，
又*定位*了行权边界，这正是树相对于公式的实际优势。

In [7]:
S0p, Kp, rp, up_, dp_, dtp = 50.0, 52.0, 0.05, 1.2, 0.8, 1.0
disc = exp(-rp * dtp)
pp = (exp(rp * dtp) - dp_) / (up_ - dp_)
show("exp(r dt)", exp(rp * dtp), 1.051271)
show("p", pp, 0.6282)

pay = lambda s: max(Kp - s, 0.0)
D, E, F = pay(S0p * up_ * up_), pay(S0p * up_ * dp_), pay(S0p * dp_ * dp_)
print("终端收益 D, E, F = %.0f, %.0f, %.0f" % (D, E, F))

cont_B = disc * (pp * D + (1 - pp) * E)
val_B = max(cont_B, pay(S0p * up_))
show("node B continuation", cont_B, 1.4148)

cont_C = disc * (pp * E + (1 - pp) * F)
val_C = max(cont_C, pay(S0p * dp_))
show("node C continuation", cont_C, 9.4639)
show("node C value (exercise wins)", val_C, 12.0)

amer = disc * (pp * val_B + (1 - pp) * val_C)
euro = disc * (pp * cont_B + (1 - pp) * cont_C)
show("American put", amer, 5.0896)
show("European put", euro, 4.1927)
show("early exercise premium", amer - euro, 0.897)

exp(r dt)                                        1.051271   book     1.0513   ok
p                                                0.628178   book     0.6282   ok
终端收益 D, E, F = 0, 4, 20
node B continuation                              1.414753   book     1.4148   ok
node C continuation                              9.463930   book     9.4639   ok
node C value (exercise wins)                    12.000000   book    12.0000   ok
American put                                     5.089632   book     5.0896   ok
European put                                     4.192654   book     4.1927   ok
early exercise premium                           0.896978   book     0.8970   ok


## 7. 由波动率构建一棵树：例 13.7

Cox–Ross–Rubinstein 的取法 $u = e^{\sigma\sqrt{\Delta t}}$、$d = 1/u$。
有两项检查每次都值得做：$0 < p < 1$，以及 $d < e^{r\Delta t} < u$。
任何一项不成立，都意味着这一步相对于波动率太长了，
而树会返回负概率而不是报错。

In [8]:
sig, dtc, rc = 0.30, 1.0 / 12, 0.04
uc = exp(sig * sqrt(dtc))
dc = 1.0 / uc
growth = exp(rc * dtc)
pc = (growth - dc) / (uc - dc)
show("u", uc, 1.09046)
show("d", dc, 0.91704)
show("exp(r dt)", growth, 1.003339)
show("p", pc, 0.49762)
print("0 < p < 1 是否成立        : %s" % (0 < pc < 1))
print("d < exp(r dt) < u 是否成立: %s" % (dc < growth < uc))

# 本章的一阶近似：p ~ 1/2 + (1/2)((r - sigma^2/2)/sigma) sqrt(dt)
approx = 0.5 + 0.5 * ((rc - sig * sig / 2) / sig) * sqrt(dtc)
show("p 的一阶近似", approx, 0.4976)


u                                                1.090463   book     1.0905   ok
d                                                0.917042   book     0.9170   ok
exp(r dt)                                        1.003339   book     1.0033   ok
p                                                0.497616   book     0.4976   ok
0 < p < 1 是否成立        : True
d < exp(r dt) < u 是否成立: True
p 的一阶近似                                          0.497594   book     0.4976   ok


## 8. 美式期货看涨期权：例 13.8

签订期货合约不花钱，因此 $q = r$，
第 11.8 节关于提前行权的论证因而失去了前提。
这里的看涨期权*确实*提前行权了，而题中没有任何股息。
只记住结论而没记住证明的读者，会给这类合约定错价。

In [9]:
F0, sigf, rf, dtf, Kf = 100.0, 0.25, 0.04, 0.25, 100.0
uf = exp(sigf * sqrt(dtf))
df_ = 1.0 / uf
pf = (1.0 - df_) / (uf - df_)                  # 期货标的对应 q = r
discf = exp(-rf * dtf)
show("u", uf, 1.133148)
show("d", df_, 0.882497)
show("p", pf, 0.4688)

show("terminal F, up-up", F0 * uf * uf, 128.40)
show("terminal F, down-down", F0 * df_ * df_, 77.88)
show("payoff, up-up", max(F0 * uf * uf - Kf, 0.0), 28.4025)

cont_up = discf * (pf * max(F0 * uf * uf - Kf, 0.0))
exer_up = F0 * uf - Kf
show("upper node continuation", cont_up, 13.1824)
show("upper node exercise", exer_up, 13.3148)
print("在上节点提前行权是否最优: %s" % (exer_up > cont_up))

val_up = max(cont_up, exer_up)
show("American call today", discf * (pf * val_up), 6.1798)
show("European call today", discf * (pf * cont_up), 6.1183)

u                                                1.133148   book     1.1331   ok
d                                                0.882497   book     0.8825   ok
p                                                0.468791   book     0.4688   ok
terminal F, up-up                              128.402542   book   128.4000   ok
terminal F, down-down                           77.880078   book    77.8800   ok
payoff, up-up                                   28.402542   book    28.4025   ok
upper node continuation                         13.182360   book    13.1824   ok
upper node exercise                             13.314845   book    13.3148   ok
在上节点提前行权是否最优: True
American call today                              6.179767   book     6.1798   ok
European call today                              6.118277   book     6.1183   ok


## 9. 收敛，以及它收敛到的极限：图 13.6

收敛图中的两年期看跌期权，在 $n$ 步的 Cox–Ross–Rubinstein 树上。
取值随 $n$ 的奇偶振荡，并向第 15 章推导的布莱克–斯科尔斯–默顿极限稳定；
图中把该极限画成虚线，而第 15 章引用的美式值，正是同一棵树在足够细时给出的。


In [10]:
from math import erf, log


def crr_put(S, K, r, sigma, T, n, american):
    """n 步 Cox-Ross-Rubinstein 树上的欧式或美式看跌期权。"""
    dt = T / n
    u = exp(sigma * sqrt(dt)); d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d); disc = exp(-r * dt)
    v = [max(K - S * u ** j * d ** (n - j), 0.0) for j in range(n + 1)]
    for i in range(n - 1, -1, -1):
        v = [disc * (p * v[j + 1] + (1 - p) * v[j]) for j in range(i + 1)]
        if american:
            v = [max(v[j], K - S * u ** j * d ** (i - j)) for j in range(i + 1)]
    return v[0]


def N(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


Sc, Kc, rr, sc, Tc = 50.0, 52.0, 0.05, 0.30, 2.0
show("欧式看跌，n = 1", crr_put(Sc, Kc, rr, sc, Tc, 1, False), 8.4503)
show("欧式看跌，n = 2", crr_put(Sc, Kc, rr, sc, Tc, 2, False), 6.2457)
show("欧式看跌，n = 50", crr_put(Sc, Kc, rr, sc, Tc, 50, False), 6.7872)
d1 = (log(Sc / Kc) + (rr + sc * sc / 2) * Tc) / (sc * sqrt(Tc))
d2 = d1 - sc * sqrt(Tc)
bs = Kc * exp(-rr * Tc) * N(-d2) - Sc * N(-d1)
show("布莱克-斯科尔斯-默顿极限", bs, 6.7601)
show("2000 步树上的美式看跌", crr_put(Sc, Kc, rr, sc, Tc, 2000, True), 7.4719)
print("奇数与偶数 n 相互交错，到 n = 50 时都仍在极限之上：",
      "%.4f (n = 49), %.4f (n = 50)" % (crr_put(Sc, Kc, rr, sc, Tc, 49, False), crr_put(Sc, Kc, rr, sc, Tc, 50, False)))


欧式看跌，n = 1                                       8.450255   book     8.4503   ok
欧式看跌，n = 2                                       6.245708   book     6.2457   ok
欧式看跌，n = 50                                      6.787222   book     6.7872   ok
布莱克-斯科尔斯-默顿极限                                    6.760140   book     6.7601   ok


2000 步树上的美式看跌                                    7.471909   book     7.4719   ok
奇数与偶数 n 相互交错，到 n = 50 时都仍在极限之上： 6.7627 (n = 49), 6.7872 (n = 50)


## 10. 提前行权边界：图 13.5

例 13.6 的看跌期权，放在一棵 1,000 步的树上，波动率取两步树所对应的值，
$\sigma = \ln(1.2/0.8)/2 = 20.3\%$。在每个日期，边界是行权与继续持有等值时的价格，
由行权占优的最高节点与其上方节点之间线性插值得到。它从略高于 40 起步，
到一年时升至约 42，只在最后几周才爬向执行价，也就是图 11.5 中的临界价格 $S^*$ 随时间的轨迹。


In [11]:
def exercise_boundary(S, K, r, sigma, T, n):
    """日期 -> 内在价值等于继续持有价值时的价格。"""
    dt = T / n
    u = exp(sigma * sqrt(dt)); d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d); disc = exp(-r * dt)
    grid = [[S * u ** j * d ** (i - j) for j in range(i + 1)] for i in range(n + 1)]
    v = [max(K - s, 0.0) for s in grid[n]]
    out = {}
    for i in range(n - 1, -1, -1):
        cont = [disc * (p * v[j + 1] + (1 - p) * v[j]) for j in range(i + 1)]
        gain = [K - grid[i][j] - cont[j] for j in range(i + 1)]      # 行权减继续持有
        v = [max(c, K - s) for c, s in zip(cont, grid[i])]
        ex = [j for j in range(i + 1) if gain[j] >= 0 and K - grid[i][j] > 0]
        if ex:
            j = max(ex)
            if j + 1 <= i:
                s0, s1, g0, g1 = grid[i][j], grid[i][j + 1], gain[j], gain[j + 1]
                out[round(i * dt, 3)] = s0 + (s1 - s0) * g0 / (g0 - g1)
            else:
                out[round(i * dt, 3)] = grid[i][j]
    return out


bnd = exercise_boundary(50.0, 52.0, 0.05, log(1.2 / 0.8) / 2, 2.0, 1000)
show("t = 0.5 处的边界", bnd[0.5], 41.0, tol=3e-3)
show("t = 1.0 处的边界", bnd[1.0], 41.9, tol=3e-3)
show("t = 1.9 处的边界", bnd[1.9], 46.8, tol=3e-3)
show("t = 1.98 处的边界", bnd[1.98], 49.2, tol=3e-3)
print("两步树在一年期日期于 40 行权、于 60 持有；",
      "细树把边界放在 %.2f" % bnd[1.0])


t = 0.5 处的边界                                    41.034991   book    41.0000   ok
t = 1.0 处的边界                                    41.939026   book    41.9000   ok
t = 1.9 处的边界                                    46.807278   book    46.8000   ok
t = 1.98 处的边界                                   49.212955   book    49.2000   ok
两步树在一年期日期于 40 行权、于 60 持有； 细树把边界放在 41.94
